# 00 — Google Colab: prepare deterministic research data

Run this notebook in **Google Colab with a CPU runtime**. It prepares Parquet inputs and stores them persistently in Google Drive. Every run is isolated by **profile + Git commit SHA** so results from different methodologies cannot mix.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/political-bias-lab.git"
PROFILE = "smoke"  # smoke -> pilot -> paper
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-lab"
REPO_DIR = "/content/political-bias-lab"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
os.chdir(REPO_DIR)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
RUN_ID = f"{PROFILE}-{GIT_SHA[:8]}"
RUN_ROOT = f"{DRIVE_ROOT}/runs/{RUN_ID}"
print("Git revision:", GIT_SHA)
print("Run ID:", RUN_ID)
print("Drive run root:", RUN_ROOT)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-core.txt"])

In [ ]:
from src.cloud import link_colab_persistent_dirs
link_colab_persistent_dirs(REPO_DIR, RUN_ROOT)
print("Persistent directories linked to:", RUN_ROOT)

## Optional real-entity inputs

For smoke/pilot pipeline validation you can use the included fictional entities. Before a paper that makes claims about **real politicians**, create `data/entities_research.csv` and `data/entity_pairs_research.csv` from the supplied templates and freeze them before the paper run.

In [ ]:
from pathlib import Path
from src.config import load_config
from src.pipeline import prepare_project
from src.reproducibility import save_run_manifest

cfg = load_config(Path(REPO_DIR)/"config/default.yaml", Path(REPO_DIR)/f"config/{PROFILE}.yaml")
stats = prepare_project(cfg, root=REPO_DIR)
save_run_manifest(Path(REPO_DIR)/"results/manifests/prepare_manifest.json", config=cfg, root=REPO_DIR)
stats

In [ ]:
expected = int(cfg["phase1"]["logical_universe"]["expected_cross_product"])
if not stats["uses_real_entities"]:
    assert stats["logical_swap_cases"] == expected
assert stats["phase3_test_items"] == 84
assert stats["phase4_cases"] == 60
print("Prepared data validation passed.")
print("Logical entity-swap universe:", f"{stats['logical_swap_cases']:,}")

### Next

Open `01_colab_generator_inference.ipynb` with a GPU runtime. Run it once for `qwen2.5-7b`, then in a fresh Colab GPU session run it for `mistral-7b-v0.3`. **Do not change the repository commit between notebooks for the same experiment.**